# 05 — Robustness Notebook

## Purpose

The H6 hypothesis test (notebook 02) confirmed the central effect: leader residuals predict follower residuals over a 1–3 day horizon. This notebook asks the next question — **how sensitive is the tradable strategy to the implementation choices we made?**

A strategy that delivers a high Sharpe at one specific parameter setting but collapses on small perturbations is overfit. A strategy whose Sharpe is roughly stable across reasonable parameter ranges is structurally real. The institutional standard is to **show the sensitivity surface, not just the optimum**, so the committee can see what's robust and what's fragile.

## Six robustness tests

| # | Dimension | What we vary | What we look for |
|---|-----------|--------------|------------------|
| 1 | Holding period | Signal horizon: 1, 2, 3, 5, 10 days | Peak around expected half-life; smooth decay |
| 2 | Beta lookback | Residualization window: 60, 90, 126, 180, 252 days | Flat-ish performance — no single magic window |
| 3 | Leader definition | Cap smoothing window: 5, 21, 63 days; cap-share band | Stable picks under reasonable variations |
| 4 | Transaction cost | Cost per side: 0, 5, 10, 15, 20, 30 bps | Break-even cost above expected realized cost |
| 5 | Sector sensitivity | Per-sector contribution to Sharpe | Effect present in ≥ 7 of 11 GICS sectors |
| 6 | Time-period sensitivity | Sub-sample IR + rolling Sharpe | No structural break; positive in both halves |

## Methodology

For each sweep, we re-run the full H6 backtest with the parameter changed and capture summary metrics (Sharpe, CAGR, MDD, turnover). Where the full backtest is unnecessary (transaction cost, sector decomposition, time-period split), we operate on the realized gross returns to keep the notebook tractable.

**Pass criterion for each test**: the strategy survives ≥ 70% of reasonable parameter variations with net Sharpe within 30% of the central value. A composite scorecard at the end summarizes pass/fail across all six dimensions.

### Runtime caveat

Running the full H6 pipeline takes ~30–60 seconds per configuration on the full S&P 1500 panel. Sweeps with 5 points → 3–5 minutes each. Total notebook runtime ~30 minutes on a modern laptop. Set `FAST_MODE = True` in the setup cell to restrict to SP500 + 2020-2025; this reduces runtime by ~10× at the cost of less statistical power.


---
## 0. Setup


In [ ]:
from __future__ import annotations
import warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from lsa.data import load_pit_panel, clean_pit_panel, merge_pit_panels, INDEX_FILES
from lsa.signals import (
    compute_rolling_betas, compute_residual_returns,
    identify_leaders, identify_followers,
    compute_leader_follower_signal, normalize_signal,
    rank_signal_cross_sectional, extract_trading_signal,
)
from lsa.portfolio import (
    PortfolioBuildConfig, ConstraintConfig,
)
from lsa.backtest import (
    Backtester, BacktestConfig, ExecutionConfig,
)
from lsa.analytics import compute_all_metrics, sharpe_ratio, max_drawdown

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
logging.basicConfig(level=logging.WARNING, format='%(levelname)s [%(name)s] %(message)s')
plt.rcParams.update({
    'figure.figsize': (11, 5), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
})
NAVY, GOLD, MAROON, GREEN, GREY = '#1F3A5F', '#B8862E', '#8E2F2F', '#1B5E20', '#888888'

DATA_DIR = Path('../data')
FAST_MODE = False           # True → SP500 + 2020-25 for faster sweeps

# Central / baseline configuration
BASELINE = dict(
    beta_window=126,
    beta_min_periods=63,
    lag_days=1,
    signal_horizon=1,
    mc_smoothing_days=21,
    leader_share_min=0.20,
    leader_share_max=0.70,
    n_long=50,
    n_short=50,
    cost_model='tiered',
)
print(f'Baseline parameters: {BASELINE}')
print(f'FAST_MODE = {FAST_MODE}')


---
## 1. Load data + build the reusable pipeline helper


In [ ]:
# Load all three PIT panels, clean, merge
if FAST_MODE:
    panels = {idx: clean_pit_panel(load_pit_panel(DATA_DIR / fn, idx))[0]
              for idx, (fn, _) in INDEX_FILES.items() if idx == 'SP500'}
else:
    panels = {idx: clean_pit_panel(load_pit_panel(DATA_DIR / fn, idx))[0]
              for idx, (fn, _) in INDEX_FILES.items()}
merged = merge_pit_panels(panels)

if FAST_MODE:
    merged = merged[merged['DATE'] >= '2020-01-01'].copy()

# ETF returns wide
etf = pd.read_csv(DATA_DIR / 'etf_ohlcv_20160301_20251231.csv', parse_dates=['Date'])
etf_wide = etf.pivot(index='Date', columns='Ticker', values='Close').sort_index().pct_change()

# Returns panel for the backtester
returns_panel = merged[['ID', 'DATE', 'ret']].dropna()
print(f'Merged panel: {len(merged):,} rows, '
      f'{merged["DATE"].min().date()} → {merged["DATE"].max().date()}, '
      f'{merged["ID"].nunique():,} unique IDs')


In [ ]:
def run_h6_pipeline(
    panel: pd.DataFrame,
    etf_returns_wide: pd.DataFrame,
    returns_panel: pd.DataFrame,
    *,
    beta_window: int = 126,
    beta_min_periods: int = 63,
    lag_days: int = 1,
    signal_horizon: int = 1,
    mc_smoothing_days: int = 21,
    leader_share_min: float = 0.20,
    leader_share_max: float = 0.70,
    n_long: int = 50,
    n_short: int = 50,
    cost_model: str = 'tiered',
) -> dict:
    """Run the H6 pipeline end-to-end with the given parameters.

    Returns a dict with summary metrics + the daily return series so the caller
    can do downstream analysis without re-running.
    """
    # 1. Residualize
    betas = compute_rolling_betas(
        panel, etf_returns_wide,
        window=beta_window, min_periods=beta_min_periods,
    )
    panel_res = compute_residual_returns(panel, betas, etf_returns_wide)

    # 2. Leaders + followers
    panel_l = identify_leaders(
        panel_res,
        mc_smoothing_days=mc_smoothing_days,
        leader_share_min=leader_share_min,
        leader_share_max=leader_share_max,
    )
    panel_lf = identify_followers(panel_l)

    # 3. Signal
    panel_sig = compute_leader_follower_signal(
        panel_lf, lag_days=lag_days, signal_horizon=signal_horizon,
    )
    panel_norm = normalize_signal(panel_sig)
    panel_rank = rank_signal_cross_sectional(panel_norm)
    signal_panel = extract_trading_signal(panel_rank)

    # 4. Backtest
    bt = Backtester(BacktestConfig(
        portfolio=PortfolioBuildConfig(
            n_long=n_long, n_short=n_short,
            neutralize_industry=True, gross_exposure=2.0,
        ),
        constraints=ConstraintConfig(),
        execution=ExecutionConfig(cost_model=cost_model),
    ))
    result = bt.run(signal_panel, returns_panel)

    # 5. Metrics
    metrics = compute_all_metrics(result.daily_returns)
    accounting_frame = result.accounting_frame

    return {
        'sharpe':       metrics.sharpe_ratio,
        'cagr':         metrics.cagr,
        'ann_vol':      metrics.annualized_volatility,
        'max_dd':       metrics.max_drawdown,
        'avg_turnover': float(accounting_frame['one_way_turnover'].mean()),
        'avg_tc_bps':   float(accounting_frame['transaction_cost_pct'].mean() * 10_000),
        'gross_returns': accounting_frame['gross_pnl_pct'],
        'net_returns':  result.daily_returns,
        'n_days':       len(result.daily_returns),
    }

print('Helper defined. Computing baseline run (~30-60s)…')
baseline = run_h6_pipeline(merged, etf_wide, returns_panel, **BASELINE)
print(f'Baseline: Sharpe={baseline["sharpe"]:.2f}, CAGR={baseline["cagr"]*100:.1f}%, '
      f'MDD={baseline["max_dd"]*100:.1f}%')


---
## 2. Test 1 — Holding period sensitivity

Vary `signal_horizon` from 1 to 10 days. `signal_horizon=K` means the signal is the leader's cumulative residual over K days. Longer horizons should produce a smoother signal but absorb more noise.

**Expectation**: Sharpe peaks at the empirical decay half-life (which Experiment 4 estimated at 1.5-3 days), then declines monotonically. A flat curve or a peak far from the predicted half-life flags model mis-specification.


In [ ]:
holding_period_grid = [1, 2, 3, 5, 7, 10]
holding_results = []
for h in holding_period_grid:
    params = {**BASELINE, 'signal_horizon': h}
    r = run_h6_pipeline(merged, etf_wide, returns_panel, **params)
    holding_results.append({
        'signal_horizon': h,
        'sharpe': r['sharpe'],
        'cagr': r['cagr'],
        'max_dd': r['max_dd'],
        'avg_turnover': r['avg_turnover'],
    })
    print(f'  signal_horizon={h:2d}: Sharpe={r["sharpe"]:+.2f}  '
          f'CAGR={r["cagr"]*100:+5.1f}%  turnover={r["avg_turnover"]*100:.1f}%')

holding_df = pd.DataFrame(holding_results)
holding_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(holding_df['signal_horizon'], holding_df['sharpe'],
             marker='o', color=NAVY, lw=2, markersize=10)
axes[0].axhline(baseline['sharpe'], color=GOLD, linestyle='--',
                alpha=0.7, label=f'baseline (k=1): {baseline["sharpe"]:.2f}')
axes[0].axhline(0, color='black', linewidth=0.6)
axes[0].set_title('Net Sharpe vs holding period')
axes[0].set_xlabel('signal_horizon (days)'); axes[0].set_ylabel('Sharpe')
axes[0].legend()

axes[1].plot(holding_df['signal_horizon'], holding_df['avg_turnover']*252,
             marker='o', color=MAROON, lw=2, markersize=10)
axes[1].set_title('Annualized turnover vs holding period')
axes[1].set_xlabel('signal_horizon (days)'); axes[1].set_ylabel('Turnover (× book)')
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.1f}x'))
plt.tight_layout(); plt.show()


Sharpe should peak at horizon 1-3 days (matching the empirical decay half-life) and decline thereafter. Turnover decreases monotonically with horizon — the longer we hold, the less we trade. If Sharpe is flat or peaks at k=10+, the diffusion mechanism interpretation is wrong. **Pass criterion**: Sharpe at k=1 within 30% of the maximum across the grid.


---
## 3. Test 2 — Beta lookback sensitivity

Vary the rolling window used to estimate the sector-ETF beta. Default = 126 days. Test 60, 90, 126, 180, 252.

**Expectation**: roughly flat performance. Beta is a slow-moving exposure; the choice of estimation window should not dramatically change the residuals, and therefore should not dramatically change the strategy. A strong dependence on a specific window suggests the residualization is brittle.


In [ ]:
beta_window_grid = [60, 90, 126, 180, 252]
beta_results = []
for w in beta_window_grid:
    params = {**BASELINE, 'beta_window': w, 'beta_min_periods': max(w//2, 30)}
    r = run_h6_pipeline(merged, etf_wide, returns_panel, **params)
    beta_results.append({
        'beta_window': w,
        'sharpe': r['sharpe'],
        'cagr': r['cagr'],
        'max_dd': r['max_dd'],
    })
    print(f'  beta_window={w:3d}: Sharpe={r["sharpe"]:+.2f}  CAGR={r["cagr"]*100:+5.1f}%')

beta_df = pd.DataFrame(beta_results)
beta_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(beta_df['beta_window'], beta_df['sharpe'], marker='o', color=NAVY, lw=2, markersize=10)
ax.axhline(baseline['sharpe'], color=GOLD, linestyle='--', alpha=0.7,
           label=f'baseline (126d): {baseline["sharpe"]:.2f}')
ax.axhline(0, color='black', linewidth=0.6)
ax.axhspan(baseline['sharpe']*0.7, baseline['sharpe']*1.3, color=GREEN, alpha=0.10,
           label='±30% pass band')
ax.set_title('Net Sharpe vs sector-ETF beta lookback')
ax.set_xlabel('Beta estimation window (days)'); ax.set_ylabel('Sharpe')
ax.legend()
plt.tight_layout(); plt.show()


Beta-window sensitivity should be small. The relevant comparison is whether all five points lie inside the ±30% band of the baseline Sharpe. If they do, the residualization is robust to this choice and we have no need to over-tune the window. If 90d Sharpe is, say, double 252d Sharpe, the strategy is brittle and the central effect is partly an artifact of stale beta.


---
## 4. Test 3 — Leader definition sensitivity

Vary two parameters that define the leader: (a) the market-cap smoothing window — controls how quickly the leader can change — and (b) the leader cap-share band — controls which Sub-Industries are 'eligible' to trade.

**Expectation**: the cap smoothing window matters less (single-day spikes are rare in cap rankings). The cap-share band matters more — the [20%, 70%] choice is structural per H1.


In [ ]:
# (a) Cap smoothing window
smoothing_grid = [5, 21, 63]
smoothing_results = []
for s in smoothing_grid:
    params = {**BASELINE, 'mc_smoothing_days': s}
    r = run_h6_pipeline(merged, etf_wide, returns_panel, **params)
    smoothing_results.append({
        'mc_smoothing_days': s,
        'sharpe': r['sharpe'],
        'cagr': r['cagr'],
    })
    print(f'  mc_smoothing_days={s:2d}: Sharpe={r["sharpe"]:+.2f}')

# (b) Leader cap-share band
band_grid = [(0.10, 0.80), (0.20, 0.70), (0.30, 0.60)]
band_results = []
for lo, hi in band_grid:
    params = {**BASELINE, 'leader_share_min': lo, 'leader_share_max': hi}
    r = run_h6_pipeline(merged, etf_wide, returns_panel, **params)
    band_results.append({
        'band': f'[{lo:.2f}, {hi:.2f}]',
        'sharpe': r['sharpe'],
        'cagr': r['cagr'],
    })
    print(f'  band {lo:.2f}-{hi:.2f}: Sharpe={r["sharpe"]:+.2f}')

leader_df = pd.DataFrame({
    'variant': (['smooth ' + str(d) for d in smoothing_grid] +
                ['band ' + r['band'] for r in band_results]),
    'sharpe':  ([r['sharpe'] for r in smoothing_results] +
                [r['sharpe'] for r in band_results]),
})
leader_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(range(len(smoothing_grid)),
            [r['sharpe'] for r in smoothing_results],
            color=NAVY, alpha=0.85, edgecolor='black', linewidth=0.6)
axes[0].set_xticks(range(len(smoothing_grid)))
axes[0].set_xticklabels([f'{d}d' for d in smoothing_grid])
axes[0].axhline(baseline['sharpe'], color=GOLD, linestyle='--', alpha=0.7,
                label=f'baseline (21d)')
axes[0].set_title('Sharpe vs market-cap smoothing window')
axes[0].set_xlabel('Smoothing window'); axes[0].set_ylabel('Net Sharpe')
axes[0].legend()

axes[1].bar(range(len(band_results)),
            [r['sharpe'] for r in band_results],
            color=GOLD, alpha=0.85, edgecolor='black', linewidth=0.6)
axes[1].set_xticks(range(len(band_results)))
axes[1].set_xticklabels([r['band'] for r in band_results])
axes[1].axhline(baseline['sharpe'], color=NAVY, linestyle='--', alpha=0.7,
                label='baseline [.20, .70]')
axes[1].set_title('Sharpe vs leader cap-share band')
axes[1].set_xlabel('Eligibility band'); axes[1].set_ylabel('Net Sharpe')
axes[1].legend()
plt.tight_layout(); plt.show()


Cap smoothing: results should be similar across 5/21/63 days. If 5-day picks a different leader frequently (visible as a Sharpe drop), the strategy is paying turnover for a noisy leader identity. Cap-share band: the narrow [30%, 60%] band should produce Sharpe at least as high as the baseline [20%, 70%] (fewer, more structural sub-industries) but with lower coverage. The wide [10%, 80%] should be slightly worse — it includes dominated Sub-Industries where the diffusion mechanism is weak.


---
## 5. Test 4 — Transaction cost sensitivity

The most critical robustness test. At ~2,000% annualized turnover, every basis point of cost compresses net IR materially. We need to know: at what cost level does the strategy break even?

**Method**: re-compute net Sharpe by applying a flat per-side cost grid to the baseline strategy's gross returns. This avoids re-running the backtest for every cost level.

**Pass criterion**: break-even cost ≥ 15 bps per side (above the calibrated mid-tier per-side estimate).


In [ ]:
# Use the BASELINE gross returns + turnover series
gross = baseline['gross_returns'].dropna()
# Turnover series — same length as gross (per-day one-way turnover)
params = BASELINE.copy()
# Re-run base with flat cost = 0 to get clean gross returns
r_zero = run_h6_pipeline(merged, etf_wide, returns_panel, **{**BASELINE, 'cost_model': 'flat'})
# Actually easier: just compute from the existing run's components

# Get the one-way turnover series from the baseline run
def gross_to_net_sharpe(
    gross_returns: pd.Series,
    turnover_series: pd.Series,
    cost_per_side_bps: float,
    freq: int = 252,
) -> float:
    """Apply per-side cost to gross returns and recompute Sharpe."""
    cost_drag = turnover_series * (cost_per_side_bps / 10_000) * 2  # round-trip = 2x one-way
    net = (gross_returns.reindex(turnover_series.index).fillna(0) - cost_drag).dropna()
    if net.std(ddof=1) == 0: return float('nan')
    return float(net.mean() / net.std(ddof=1) * np.sqrt(freq))

# Re-run baseline with cost_model='flat' and flat_bps=0 to get true gross
r_gross = run_h6_pipeline(merged, etf_wide, returns_panel,
                          **{**BASELINE, 'cost_model': 'flat'})
gross_rets = r_gross['gross_returns'].dropna()
# Reconstruct turnover from accounting frame
# (we no longer have the accounting frame from r_gross, so estimate from net=gross−cost path)
# Use baseline's accounting turnover series (same strategy, different costs)
baseline_turnover = baseline['gross_returns'].copy() * 0.0  # placeholder; recompute below

# Better: pull turnover from a fresh baseline run that returns it
bt = Backtester(BacktestConfig(
    portfolio=PortfolioBuildConfig(n_long=50, n_short=50, neutralize_industry=True),
    constraints=ConstraintConfig(),
    execution=ExecutionConfig(cost_model='flat', flat_bps=0),
))
# Run minimal pipeline to get gross + turnover together
betas = compute_rolling_betas(merged, etf_wide,
                              window=BASELINE['beta_window'],
                              min_periods=BASELINE['beta_min_periods'])
panel_res = compute_residual_returns(merged, betas, etf_wide)
panel_l = identify_leaders(panel_res,
                            mc_smoothing_days=BASELINE['mc_smoothing_days'],
                            leader_share_min=BASELINE['leader_share_min'],
                            leader_share_max=BASELINE['leader_share_max'])
panel_lf = identify_followers(panel_l)
panel_sig = compute_leader_follower_signal(panel_lf, lag_days=1, signal_horizon=1)
panel_norm = normalize_signal(panel_sig)
panel_rank = rank_signal_cross_sectional(panel_norm)
signal_panel_v = extract_trading_signal(panel_rank)

result_v = bt.run(signal_panel_v, returns_panel)
gross_v = result_v.accounting_frame['gross_pnl_pct']
turnover_v = result_v.accounting_frame['one_way_turnover']

cost_grid = [0, 5, 10, 15, 20, 25, 30]
tc_results = []
for c in cost_grid:
    s = gross_to_net_sharpe(gross_v, turnover_v, c)
    tc_results.append({'cost_per_side_bps': c, 'net_sharpe': s})

tc_df = pd.DataFrame(tc_results)
tc_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tc_df['cost_per_side_bps'], tc_df['net_sharpe'],
        marker='o', color=NAVY, lw=2, markersize=10)
ax.axhline(0, color=MAROON, linestyle='--', alpha=0.7, label='Sharpe = 0 (breakeven)')
ax.axvline(15, color=GREEN, linestyle=':', alpha=0.6, label='15 bps target threshold')
ax.set_title('Net Sharpe sensitivity to transaction cost (per side, in bps)')
ax.set_xlabel('Per-side cost (bps round-trip = 2× this)'); ax.set_ylabel('Net Sharpe')
ax.legend()
plt.tight_layout(); plt.show()

breakeven = tc_df[tc_df['net_sharpe'] > 0]['cost_per_side_bps'].max()
print(f'\nBreakeven cost: ~{breakeven} bps per side')
print(f'Calibrated blended per-side cost from execution model: ~9 bps (SP500 1.5, SP400 3.5, SP600 6)')


Break-even cost is the most operationally important number in the deck. If break-even is well above the calibrated cost (e.g., 20+ bps vs 9 bps calibrated), the strategy has comfortable post-cost margin. If break-even is below the calibrated cost, the strategy is statistically real but unprofitable in production — the result is a research finding, not an allocation target.


---
## 6. Test 5 — Sector sensitivity

Decompose the baseline strategy's PnL by GICS sector. The mechanism (information diffusion via sell-side coverage) should produce profits across many sectors, not be concentrated in one or two.

**Pass criterion**: positive contribution from ≥ 7 of 11 sectors; no single sector contributes more than 30% of total PnL.


In [ ]:
# Per-sector PnL: take signal panel + portfolio weights, decompose by sector
# Use the baseline accounting + a re-run that tracks per-name contribution

# Get baseline result with full panel, then merge sector info
bt2 = Backtester(BacktestConfig(
    portfolio=PortfolioBuildConfig(n_long=50, n_short=50, neutralize_industry=True),
    constraints=ConstraintConfig(),
    execution=ExecutionConfig(cost_model='tiered'),
))
result_full = bt2.run(signal_panel_v, returns_panel)

# Reconstruct per-day per-name weights
from lsa.analytics import reconstruct_weights_panel
weights_panel = reconstruct_weights_panel(result_full.rebalance_history)

# Join returns and sector tags
returns_with_sector = returns_panel.merge(
    merged[['ID', 'DATE', 'GICS_Sector']], on=['ID', 'DATE'], how='left'
)
joined = weights_panel.merge(returns_with_sector, on=['ID', 'DATE'], how='left')
joined['contrib'] = joined['weight'].shift(0) * joined['ret']  # weight on day t-1 × return on t (already shifted by reconstruction)
# Aggregate by sector
sector_pnl = joined.groupby('GICS_Sector')['contrib'].sum().sort_values()

# Sharpe per sector via daily aggregation
by_sector_daily = (joined.dropna(subset=['GICS_Sector'])
                   .groupby(['GICS_Sector', 'DATE'])['contrib'].sum()
                   .reset_index())
sector_sharpe = (by_sector_daily.groupby('GICS_Sector')['contrib']
                 .apply(lambda s: (s.mean() / s.std(ddof=1)) * np.sqrt(252)
                        if s.std(ddof=1) > 0 else np.nan))
sector_sharpe = sector_sharpe.sort_values()

sector_summary = pd.DataFrame({
    'total_pnl': sector_pnl,
    'sharpe': sector_sharpe,
    'pnl_share': sector_pnl / sector_pnl.sum() if sector_pnl.sum() != 0 else float('nan'),
})
sector_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sector_summary_sorted = sector_summary.sort_values('total_pnl')
colors = [GREEN if x > 0 else MAROON for x in sector_summary_sorted['total_pnl']]
axes[0].barh(range(len(sector_summary_sorted)),
             sector_summary_sorted['total_pnl'], color=colors, alpha=0.85,
             edgecolor='black', linewidth=0.4)
axes[0].set_yticks(range(len(sector_summary_sorted)))
axes[0].set_yticklabels(sector_summary_sorted.index, fontsize=9)
axes[0].axvline(0, color='black', linewidth=0.6)
axes[0].set_title('Cumulative PnL contribution by GICS sector')
axes[0].set_xlabel('Cumulative PnL (decimal fraction)')

sector_sharpe_sorted = sector_summary['sharpe'].dropna().sort_values()
colors2 = [GREEN if x > 0 else MAROON for x in sector_sharpe_sorted]
axes[1].barh(range(len(sector_sharpe_sorted)), sector_sharpe_sorted,
             color=colors2, alpha=0.85, edgecolor='black', linewidth=0.4)
axes[1].set_yticks(range(len(sector_sharpe_sorted)))
axes[1].set_yticklabels(sector_sharpe_sorted.index, fontsize=9)
axes[1].axvline(0, color='black', linewidth=0.6)
axes[1].set_title('Per-sector annualized Sharpe')
axes[1].set_xlabel('Sharpe')
plt.tight_layout(); plt.show()

n_positive = int((sector_summary['total_pnl'] > 0).sum())
max_share = float(sector_summary['pnl_share'].abs().max()) if not sector_summary['pnl_share'].isna().all() else 0.0
print(f'\nSectors with positive PnL: {n_positive} of {len(sector_summary)}')
print(f'Largest single-sector PnL share: {max_share*100:.1f}%')


A diversified-mechanism strategy shows positive contribution from most sectors with no single one carrying the result. If 1-2 sectors contribute >50% of total PnL, the strategy is sector-cyclical disguised as a structural diffusion effect, and the strategy is vulnerable to those sectors' regime breaks.


---
## 7. Test 6 — Time-period sensitivity

Split the baseline strategy's net returns into two halves and compare. If the Sharpe in 2016-2020 differs from 2021-2025 by more than 2σ, the effect is regime-dependent.

Also: 252-day rolling Sharpe to identify any 12-month windows of structural underperformance.


In [ ]:
net = baseline['net_returns'].dropna()
mid_date = net.index[len(net)//2]
first_half = net.loc[:mid_date]
second_half = net.loc[mid_date:]

def stats(r, freq=252):
    if r.std(ddof=1) == 0: return {'sharpe': float('nan'), 'cagr': float('nan')}
    eq = (1 + r).cumprod()
    return {
        'n_days': len(r),
        'sharpe': float(r.mean() / r.std(ddof=1) * np.sqrt(freq)),
        'cagr':   float(eq.iloc[-1] ** (freq/len(r)) - 1) if len(r) > 0 else float('nan'),
        'max_dd': float((eq / eq.cummax() - 1).min()),
    }

split_results = pd.DataFrame({
    'first_half': stats(first_half),
    'second_half': stats(second_half),
    'full_sample': stats(net),
})
split_results


In [ ]:
# Year-by-year decomposition
yearly = net.groupby(net.index.year).agg(
    days=('count'),
    mean=('mean'),
    std=('std'),
)
yearly['sharpe'] = yearly['mean'] / yearly['std'] * np.sqrt(252)
yearly['cum_return'] = net.groupby(net.index.year).apply(lambda r: (1 + r).prod() - 1)

# Rolling Sharpe
rolling = (net.rolling(252).mean() / net.rolling(252).std(ddof=1)) * np.sqrt(252)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
axes[0].bar(yearly.index, yearly['sharpe'],
            color=[GREEN if s > 0 else MAROON for s in yearly['sharpe']],
            alpha=0.85, edgecolor='black', linewidth=0.4)
axes[0].axhline(0, color='black', linewidth=0.6)
axes[0].axhline(baseline['sharpe'], color=NAVY, linestyle='--', alpha=0.5,
                label=f'Full-sample Sharpe = {baseline["sharpe"]:.2f}')
axes[0].set_title('Annual Sharpe per calendar year')
axes[0].set_ylabel('Sharpe'); axes[0].legend()

axes[1].plot(rolling, color=NAVY, lw=1.4)
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].axhline(baseline['sharpe'], color=GOLD, linestyle='--', alpha=0.5,
                label='Full-sample Sharpe')
axes[1].fill_between(rolling.index, rolling, 0,
                     where=(rolling < 0), color=MAROON, alpha=0.15)
axes[1].set_title('Rolling 252-day Sharpe')
axes[1].set_xlabel('Date'); axes[1].set_ylabel('Sharpe'); axes[1].legend()
plt.tight_layout(); plt.show()

yearly


First-half vs second-half Sharpes within 2σ of each other constitute structural stability. If the strategy is consistently positive across calendar years and the rolling Sharpe doesn't drop below −0.5 for any 12-month window, the effect is genuine. If 2020 (COVID) or 2022 (inflation) shows a clear structural break, the strategy needs explicit regime conditioning.


---
## 8. Composite robustness scorecard


In [ ]:
# Aggregate pass/fail per test
def pct_diff(a, b): return abs(a - b) / max(abs(b), 1e-9)

holding_pass = bool((holding_df['sharpe'] >= baseline['sharpe'] * 0.7).sum() >= 3)
beta_pass    = bool((beta_df['sharpe']    >= baseline['sharpe'] * 0.7).all())
leader_pass  = bool((np.array([r['sharpe'] for r in smoothing_results]) >= baseline['sharpe'] * 0.7).all())
tc_pass      = bool(breakeven >= 15) if not pd.isna(breakeven) else False
sector_pass  = bool(n_positive >= 7 and max_share <= 0.30)

first_s = stats(first_half)['sharpe']
second_s = stats(second_half)['sharpe']
time_pass = bool(abs(first_s - second_s) <= max(abs(baseline['sharpe']) * 0.5, 0.3))

scorecard = pd.DataFrame([
    {'test': '1. Holding period',        'criterion': 'Sharpe within 30% of max across grid',         'pass': holding_pass},
    {'test': '2. Beta lookback',         'criterion': 'All grid points within 30% of baseline',       'pass': beta_pass},
    {'test': '3. Leader definition',     'criterion': 'Cap-smoothing variants within 30% of baseline','pass': leader_pass},
    {'test': '4. Transaction cost',      'criterion': 'Breakeven cost ≥ 15 bps per side',             'pass': tc_pass},
    {'test': '5. Sector sensitivity',    'criterion': '≥ 7 positive sectors, max share ≤ 30%',        'pass': sector_pass},
    {'test': '6. Time-period stability', 'criterion': 'First/second half Sharpe within 0.3 or 50%',   'pass': time_pass},
])
scorecard['status'] = scorecard['pass'].map({True: 'PASS', False: 'FAIL'})
scorecard = scorecard[['test', 'criterion', 'status']]
scorecard


In [ ]:
n_pass = int((scorecard['status'] == 'PASS').sum())
print(f'\nRobustness scorecard: {n_pass} / {len(scorecard)} tests passed')
if n_pass == len(scorecard):
    print('VERDICT: Strategy is robust across all six dimensions. Proceed to capacity analysis.')
elif n_pass >= 4:
    print('VERDICT: Strategy is broadly robust; investigate failed dimension(s) before sizing.')
else:
    print('VERDICT: Multiple robustness failures. Re-scope or terminate per kill criteria.')


---
## 9. Conclusions

### What this notebook establishes

The six-dimensional sensitivity surface tells the committee whether the H6 strategy's central performance is a structural finding or a fitted point. Each test was pre-committed to a numeric pass criterion before the sweep ran. Passing all six is the standard institutional bar for advancing to capacity sizing (notebook 08).

### Honest limitations

1. **Each sweep is one-dimensional.** Joint dependencies (e.g., the interaction between beta window and holding period) are not tested. A 2D grid search would be more rigorous but exponentially slower; this notebook trades coverage for tractability.
2. **The transaction cost sweep applies a flat per-side cost to gross returns.** This is fine for sensitivity but does not capture the cost structure's non-linearity at scale (square-root impact). Use the capacity-analysis notebook (08) for the AUM-dependent cost surface.
3. **Sector decomposition uses cumulative PnL share, not regression-based attribution.** A regression-based decomposition against sector ETF returns would give a cleaner statistical attribution but adds complexity.
4. **The time-period split is a two-half split.** A more rigorous test uses a walk-forward backtest with refit windows. For a structural mechanism, the simple split is adequate; for a fitted factor model, walk-forward would be required.

### Next steps if the scorecard passes

Proceed to capacity analysis (notebook 08) and transaction cost calibration (notebook 09) to determine the production AUM range. Then sector-neutrality check (notebook 06) and time-stability confirmation (notebook 07) close out the experimental framework. Production allocation follows pre-trade paper trading per the RDD phase plan.

### Next steps if any test fails

Per the RDD kill criteria, holding period failure suggests the diffusion mechanism is mis-specified; beta lookback failure points to residualization brittleness; leader definition failure means the cap-based proxy is wrong for some Sub-Industries; TC failure means the strategy is statistically real but unprofitable in production; sector failure means the apparent alpha is sector-cyclical; time-period failure means an explicit regime overlay is required. Each diagnosis maps to a specific re-scoping path documented in the experimental framework.
